<a href="https://colab.research.google.com/github/Rahmatul-Rovi/Data-Mining-Assotiation-Rule/blob/main/Lab_Report_on_Association_Rule_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
prasad22_retail_transactions_dataset_path = kagglehub.dataset_download('prasad22/retail-transactions-dataset')

print('Data source import complete.')



# Lab Report on Association Rule Mining

**Name:** Rahmatul Munim  
**Student ID:** 20234103386  
**Course:** CSE-450  
**Date:** 20 October 2026

## 1. Objective

The objective of association rule mining is to discover relationships and co-occurrence patterns among products in transaction data. It identifies products that frequently appear together in the same transaction.

In this experiment, the FP-Growth algorithm is used to discover frequent itemsets and generate association rules. The rules are evaluated using support, confidence, and lift.


In [ ]:

import os
import glob
import ast
import sys
import warnings
import subprocess
import importlib.util

import pandas as pd

# Install mlxtend only if it is not already installed.
if importlib.util.find_spec("mlxtend") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "mlxtend"
    ])

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Hide only the known Jupyter deprecation warning.
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="jupyter_client.session"
)

SAMPLE_SIZE = 100_000
RANDOM_STATE = 42
SUPPORT_THRESHOLDS = [0.01, 0.005, 0.002, 0.001]

DATASET_URL = (
    "https://www.kaggle.com/datasets/"
    "prasad22/retail-transactions-dataset"
)

print("Libraries imported successfully.")


In [ ]:

# ---------- 2. Load dataset ----------

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

if not csv_files:
    raise FileNotFoundError(
        "No CSV file found. Please add the Retail Transactions "
        "Dataset using Kaggle's Add Input option."
    )

csv_path = None
product_column = None

for file_path in csv_files:
    try:
        columns = pd.read_csv(file_path, nrows=0).columns
        column_map = {
            str(column).strip().lower(): column
            for column in columns
        }

        if "product" in column_map:
            csv_path = file_path
            product_column = column_map["product"]
            break

    except Exception:
        continue

if csv_path is None:
    raise ValueError(
        "Could not find a CSV file containing a Product column."
    )

df = pd.read_csv(csv_path, usecols=[product_column])
df.columns = ["Product"]

TOTAL_TRANSACTIONS = len(df)

print("Dataset loaded successfully.")
print("CSV file:", os.path.basename(csv_path))
print("Original transactions:", f"{TOTAL_TRANSACTIONS:,}")
print("Product column:", "Product")

# ---------- 3. Select a reproducible sample ----------

if TOTAL_TRANSACTIONS > SAMPLE_SIZE:
    df = df.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

SAMPLED_TRANSACTIONS = len(df)

print("Transactions selected for analysis:",
      f"{SAMPLED_TRANSACTIONS:,}")

# ---------- 4. Convert product strings to Python lists ----------

def parse_products(value):
    if not isinstance(value, str):
        if pd.isna(value):
            return []
        value = str(value)

    value = value.strip()

    if not value:
        return []

    try:
        products = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        products = [value]

    if isinstance(products, str):
        products = [products]

    if not isinstance(products, (list, tuple, set)):
        return []

    cleaned = []

    for product in products:
        if product is None:
            continue

        name = str(product).strip()

        if name and name not in cleaned:
            cleaned.append(name)

    return cleaned


transactions = [
    parse_products(value)
    for value in df["Product"]
]

# Remove empty baskets, if any.
transactions = [t for t in transactions if t]

if not transactions:
    raise ValueError(
        "No valid product transactions were found."
    )

N_USED = len(transactions)

all_products = {
    product
    for transaction in transactions
    for product in transaction
}

N_UNIQUE_PRODUCTS = len(all_products)

transaction_lengths = [
    len(transaction)
    for transaction in transactions
]

print("\nPreprocessing completed.")
print("Valid transactions analyzed:", f"{N_USED:,}")
print("Unique products:", N_UNIQUE_PRODUCTS)
print("Average products per transaction:",
      round(sum(transaction_lengths) / N_USED, 2))
print("Minimum products in a transaction:",
      min(transaction_lengths))
print("Maximum products in a transaction:",
      max(transaction_lengths))


In [ ]:

# ---------- 5. Boolean transaction matrix ----------

encoder = TransactionEncoder()

encoded_array = encoder.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    encoded_array,
    columns=encoder.columns_,
    dtype=bool
)

print("Boolean conversion completed.")
print("Transaction matrix shape:", basket.shape)

print("\nFirst five rows:")
display(basket.head())

print("\nExample interpretation:")
print("True  = product is present in the transaction")
print("False = product is absent from the transaction")


In [ ]:

# ---------- 6. Frequent itemset mining ----------

frequent_itemsets_by_support = {}
summary_rows = []

for min_support in SUPPORT_THRESHOLDS:
    print(
        f"Mining frequent itemsets at support "
        f"{min_support} ...",
        flush=True
    )

    itemsets = fpgrowth(
        basket,
        min_support=min_support,
        use_colnames=True,
        max_len=3
    )

    itemsets["size"] = itemsets["itemsets"].apply(len)

    frequent_itemsets_by_support[min_support] = itemsets

    counts = itemsets["size"].value_counts().to_dict()

    summary_rows.append({
        "Minimum Support": min_support,
        "1-Itemsets": counts.get(1, 0),
        "2-Itemsets": counts.get(2, 0),
        "3-Itemsets": counts.get(3, 0),
        "Total Itemsets": len(itemsets)
    })

    print(
        f"Completed: {len(itemsets)} itemsets",
        flush=True
    )

summary_table = pd.DataFrame(summary_rows)

print("\nTable 1: Frequent itemsets at different thresholds")
display(summary_table)


In [ ]:

# ---------- 7. Top frequent product pairs ----------

LOWEST_SUPPORT = min(SUPPORT_THRESHOLDS)

lowest_support_itemsets = frequent_itemsets_by_support[
    LOWEST_SUPPORT
]

pair_itemsets = lowest_support_itemsets[
    lowest_support_itemsets["size"] == 2
].copy()

pair_itemsets["Itemset"] = pair_itemsets["itemsets"].apply(
    lambda items: ", ".join(sorted(items))
)

top_pairs = (
    pair_itemsets[
        ["Itemset", "support"]
    ]
    .sort_values("support", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

print("Table 2: Top five frequent product pairs")
display(top_pairs)


In [ ]:

# ---------- 8. Association rule generation ----------

itemsets_for_rules = frequent_itemsets_by_support[
    LOWEST_SUPPORT
].drop(columns=["size"], errors="ignore")

try:
    rules = association_rules(
        itemsets_for_rules,
        metric="confidence",
        min_threshold=0.0
    )
except TypeError:
    rules = association_rules(
        itemsets_for_rules,
        num_itemsets=len(basket),
        metric="confidence",
        min_threshold=0.0
    )

if rules.empty:
    raise ValueError(
        "No association rules were generated. "
        "Check the frequent itemsets and support threshold."
    )

rules["Antecedent"] = rules["antecedents"].apply(
    lambda items: ", ".join(sorted(items))
)

rules["Consequent"] = rules["consequents"].apply(
    lambda items: ", ".join(sorted(items))
)

rules = rules.sort_values(
    "lift",
    ascending=False
).reset_index(drop=True)

MAX_LIFT = rules["lift"].max()
MEAN_LIFT = rules["lift"].mean()

print("Total association rules:", f"{len(rules):,}")
print("Maximum lift:", round(MAX_LIFT, 6))
print("Mean lift:", round(MEAN_LIFT, 6))

top_rules = rules[
    [
        "Antecedent",
        "Consequent",
        "support",
        "confidence",
        "lift"
    ]
].head(5).copy()

top_rules[["support", "confidence", "lift"]] = (
    top_rules[["support", "confidence", "lift"]].round(6)
)

print("\nTable 3: Top association rules")
display(top_rules)


In [ ]:

# ---------- 9. Discussion ----------

print("DISCUSSION")
print("=" * 60)

print(
    f"The analysis used {N_USED:,} transactions and "
    f"identified {N_UNIQUE_PRODUCTS} unique products."
)

print(
    f"At minimum support {LOWEST_SUPPORT}, "
    f"{len(itemsets_for_rules)} frequent itemsets were found."
)

print(
    f"The algorithm generated {len(rules):,} association rules."
)

print("Maximum lift:", round(MAX_LIFT, 3))
print("Mean lift:", round(MEAN_LIFT, 3))

if MAX_LIFT > 1:
    print(
        "At least one rule shows positive association "
        "because its lift is greater than 1."
    )
elif MAX_LIFT == 1:
    print(
        "The strongest rule has a lift of 1, indicating "
        "no association beyond the baseline for that rule."
    )
else:
    print(
        "All generated rules have lift below 1."
    )

print("\nBusiness interpretation:")
print(
    "Retailers may investigate high-lift product pairs "
    "for cross-selling, recommendations, or product placement."
)

print("\nLimitations:")
print(
    "The analysis uses a random sample rather than all transactions. "
    "Association rules describe co-occurrence, not causation. "
    "Low support or confidence can limit business usefulness, "
    "and results depend on the selected support thresholds."
)
